# Orchestrating a Team of Agents

### One safety desk, eight vehicles, 3,976 owner complaints — and the question of how many agents it takes

A single agent with good tools is a strong baseline. It is also, very often, the right
answer. This notebook is about the cases where it is not, and about the thing that replaces
it — not "more agents", but an **architecture**: who decides, who sees what, who checks whom,
and when the whole thing stops.

Every part ends with numbers. By the end there is a table of them, and the last part is
about reading that table and deciding, for your own problem, how many agents you actually need.

## The desk

**Halyard Analytics** monitors vehicle defect reports for insurers and fleet operators. Its
field-safety desk is called **Canary**, and every Monday it has one job:

> For each vehicle on the watchlist, find the three components whose owner complaints look most
> dangerous, say in one sentence what is failing in each, back it with the complaint count and
> report numbers, and make a call — **escalate**, **monitor**, or **close**.

The brief goes to a safety engineer who signs it. If it is wrong, an insurer prices risk on a
defect that isn't there, or misses one that is.

In [1]:
#%pip install -q langchain langgraph langchain-openai langgraph-checkpoint-sqlite openai-agents pandas ipython-autotime

In [2]:
%load_ext autotime

time: 87 µs (started: 2026-09-26 20:49:59 +05:30)


In [47]:
import json
import operator
import sqlite3
import time
import warnings
from typing import Annotated, Literal, TypedDict

import pandas as pd
from pydantic import BaseModel, Field

# Two models, on purpose. Eight analysts read narratives in parallel — that is extraction
# work, and the small model does it. Routing, writing and verifying are judgement, and get
# the larger one. Spending unevenly is one of the things having a team buys you.
WORKER = "gpt-5-nano"
JUDGE = "gpt-5-mini"

# Everything in this notebook runs at the same reasoning effort, so that no comparison
# between two designs is really a comparison between two settings.
EFFORT = "high"

# Structured output makes pydantic grumble about a `parsed` field on every single call.
# The warning says nothing about this notebook's correctness.
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic.main")

time: 403 µs (started: 2026-09-26 22:12:48 +05:30)


In [26]:
import os
import re
from dotenv import load_dotenv


pd.set_option("display.max_columns", None)       # show every column of a wide table
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)


#def pretty_print(*args, width=95):
#    """Reflow long prose to `width`, but leave tables and SQL output untouched."""
#    text = " ".join(str(a) for a in args)
#    if "\n" in text.strip("\n") or re.search(r"\S  +\S", text):
#        print(text)
#    else:
#        print(textwrap.fill(text.strip(), width=width))


load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — check the openai_key.env path."


time: 1.28 ms (started: 2026-09-26 21:17:29 +05:30)


### The field reports

**Four tables, built from NHTSA's public complaint and recall feeds.**

| table | rows | what it holds |
|---|---|---|
| `watchlist` | 8 | The vehicles the desk covers, with their complaint and recall counts |
| `complaints` | 3,976 | Owner-filed reports: the narrative, plus crash / fire / injury / death flags |
| `complaint_components` | 5,710 | One row per (complaint, component) — the tallies come from here |
| `recalls` | 99 | Campaigns actually opened, with the component each one covers |

- **The narratives are why a language model is in this system at all,** and why it cannot all
  fit in one place: 2.36 million characters, about 591,000 tokens.
- **Every read in this notebook goes through `sql()`, and `sql()` opens the file read-only.** No
  agent can change the data it is scored against.

In [4]:
DB_PATH = "field_reports.db"


def sql(query, params=(), limit=50):
    """Run one query against the field reports. Returns up to `limit` rows, each a dict.

    `mode=ro` opens the file read-only. A plain sqlite3.connect(path) would happily run a
    DROP TABLE for an agent, so "read-only" has to live in the connection, not in a docstring.
    """
    con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row  # rows that know their column names
    rows = [dict(row) for row in con.execute(query, params).fetchmany(limit)]
    con.close()
    return rows

time: 379 µs (started: 2026-09-26 20:49:59 +05:30)


In [5]:
# The watchlist: all eight vehicles the desk covers.
WATCHLIST = [w["vehicle_id"]
             for w in sql("SELECT vehicle_id FROM watchlist ORDER BY complaint_count DESC")]

total = sql("SELECT COUNT(*) n, SUM(LENGTH(narrative)) chars FROM complaints")[0]
print(f"{total['n']} complaints, {total['chars'] / 1e6:.2f}M characters, "
      f"~{total['chars'] / 4000:.0f}k tokens")

pd.DataFrame(sql("SELECT * FROM watchlist ORDER BY complaint_count DESC"))

3976 complaints, 2.36M characters, ~591k tokens


,vehicle_id,make,model,model_year,complaint_count,recall_count
0,ford-f-150-2021,ford,f-150,2021,1006,29
1,honda-accord-2019,honda,accord,2019,685,6
2,tesla-model-3-2021,tesla,model 3,2021,660,22
3,toyota-rav4-2020,toyota,rav4,2020,634,6
4,jeep-grand-cherokee-2021,jeep,grand cherokee,2021,403,12
5,nissan-rogue-2021,nissan,rogue,2021,286,10
6,chevrolet-bolt-ev-2020,chevrolet,bolt ev,2020,172,8
7,hyundai-elantra-2021,hyundai,elantra,2021,130,6


time: 9.09 ms (started: 2026-09-26 20:49:59 +05:30)


In [7]:
pd.options.display.max_colwidth = None

time: 188 µs (started: 2026-09-26 20:50:08 +05:30)


In [8]:
# complaints: the five newest on one vehicle, narrative cut to 70 characters for display.
pd.DataFrame(sql("""
    SELECT odi_number, vehicle_id, date_filed, components, crash, fire, injuries, deaths,
           was_masked, narrative 
      FROM complaints
     WHERE vehicle_id = 'chevrolet-bolt-ev-2020'
     ORDER BY odi_number DESC""", limit=5))

,odi_number,vehicle_id,date_filed,components,crash,fire,injuries,deaths,was_masked,narrative
0,11755357,chevrolet-bolt-ev-2020,08/06/2026,STEERING,0,0,0,0,0,"Steering rack is starting to fail at under 47000 miles, car pulls to right during acceleration and pulls left when decelerating. Steering wheel does not return to the center and a slight left pull on the wheel has to be maintained at all times so the car doesn't merge right into traffic. It is available upon inspection request. I almost didn't realize it pulling to the right and almost merged into another car on the highway. It has not been confirmed yet by a third party. It has not been inspected yet. No warning lights or messages occured when it started happening."
1,11748422,chevrolet-bolt-ev-2020,07/06/2026,FUEL/PROPULSION SYSTEM,0,0,0,0,1,GM installed Advanced Monitoring Software on the car rather than replace the battery. The software detected a battery fault. Under the battery [REDACTED] if a fault is detected the battery is replaced free of charge. GM is refusing to replace the battery. Now the car will not move.
2,11745256,chevrolet-bolt-ev-2020,06/19/2026,"ELECTRICAL SYSTEM,FUEL/PROPULSION SYSTEM",0,0,0,0,1,"General Motors is using the arbitrary 6,213-mile tracking window of [REDACTED] 944 to deny coverage for a diagnosed Cell Section 3 hardware failure at 123,582 miles. However, this vehicle is directly impacted by the failure parameters outlined in the expanded GM Safety [REDACTED] N242470160 / N242470162, where GM openly acknowledges to federal regulators that the advanced diagnostic software has failed to properly detect defective battery modules. By denying a physical battery replacement when a manufacturing defect is caught during a mandatory corporate software reconfiguration, GM is violating the spirit and legal intent of their ongoing federal safety [REDACTED] campaigns."
3,11742825,chevrolet-bolt-ev-2020,06/08/2026,STEERING,0,0,0,0,0,Steering stiffened over several weeks. Then steering wheel would not automatically return to center while turning while driving. Diagnosed by service as a fault with steering column. This is apparently very common in Chevy Bolts.
4,11742485,chevrolet-bolt-ev-2020,06/06/2026,STRUCTURE,0,0,0,0,0,"After putting washer fluid in, my son closed the hood. The hood did not latch. He tried again but it still didn’t latch. He lifted the hood to find the hood striker had come off the hood. There was nothing to hold the hood closed. I’ve included photos of the striker welds. There is no rust or tearing of the welds. It just separated. It was a bad weld."


time: 3.39 ms (started: 2026-09-26 20:50:08 +05:30)


In [9]:
# complaint_components: the same five complaints, one row per component they name. The one
# that names two components counts once under each.
pd.DataFrame(sql("""
    SELECT *
      FROM complaint_components
     WHERE odi_number IN (SELECT odi_number FROM complaints
                           WHERE vehicle_id = 'chevrolet-bolt-ev-2020'
                           ORDER BY odi_number DESC LIMIT 5)
     ORDER BY odi_number DESC"""))

,odi_number,vehicle_id,component
0,11755357,chevrolet-bolt-ev-2020,STEERING
1,11748422,chevrolet-bolt-ev-2020,FUEL/PROPULSION SYSTEM
2,11745256,chevrolet-bolt-ev-2020,FUEL/PROPULSION SYSTEM
3,11745256,chevrolet-bolt-ev-2020,ELECTRICAL SYSTEM
4,11742825,chevrolet-bolt-ev-2020,STEERING
5,11742485,chevrolet-bolt-ev-2020,STRUCTURE


time: 2.96 ms (started: 2026-09-26 20:51:16 +05:30)


In [11]:
# recalls: `component` is NHTSA's full path; `component_head` is its first part, the same
# vocabulary the complaints use, so it is what the scoreboard matches on.
pd.DataFrame(sql("""
    SELECT campaign_number, vehicle_id, report_date, component_head, component,
           consequence
      FROM recalls
     WHERE vehicle_id = 'chevrolet-bolt-ev-2020'
     ORDER BY campaign_number""", limit=5))

,campaign_number,vehicle_id,report_date,component_head,component,consequence
0,20V184000,chevrolet-bolt-ev-2020,26/03/2020,LATCHES/LOCKS/LINKAGES,LATCHES/LOCKS/LINKAGES:DOORS:LATCH,"If the rear door opens while driving, or the door handle fails to open the rear door, there is an increased risk of injury to the rear passengers."
1,20V808000,chevrolet-bolt-ev-2020,22/12/2020,SERVICE BRAKES,"SERVICE BRAKES, HYDRAULIC:FOUNDATION COMPONENTS:DISC:CALIPER","If a brake caliper fractures and brake fluid is lost, the vehicle may experience reduced brake performance, increasing the risk of a crash."
2,20V811000,chevrolet-bolt-ev-2020,23/12/2020,SEAT BELTS,SEAT BELTS,"If a seat belt assembly is not properly attached to the vehicle, the seat belt may not properly restrain an occupant in the event of a crash, increasing the risk of injury."
3,21V650000,chevrolet-bolt-ev-2020,20/08/2021,ELECTRICAL SYSTEM,ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY,A battery fire increases the risk of injury.
4,22V930000,chevrolet-bolt-ev-2020,15/12/2022,STRUCTURE,STRUCTURE:BODY:ROOF AND PILLARS,A vehicle fire can increase the risk of injury.


time: 3.08 ms (started: 2026-09-26 20:52:19 +05:30)


In [12]:
def narratives(vehicle_id, component=None, limit=60, chars=600):
    """The complaint narratives for one vehicle, worst reported harm first and newest first
    within the same harm, each cut to `chars`."""
    query = """SELECT odi_number, date_filed, components, crash, fire, injuries, deaths,
                      SUBSTR(narrative, 1, ?) AS narrative
                 FROM complaints
                WHERE vehicle_id = ?"""
    params = [chars, vehicle_id]
    if component:
        query += """ AND odi_number IN (SELECT odi_number FROM complaint_components
                                         WHERE component = ?)"""
        params.append(component)
    # odi_number, not date_filed: the date is month/day/year text and would sort by month.
    query += " ORDER BY deaths DESC, injuries DESC, fire DESC, odi_number DESC"
    return sql(query, params, limit)


def components(vehicle_id):
    """Every component named on this vehicle's complaints, most complaints first."""
    return sql("""
        SELECT cc.component,
               COUNT(*) AS complaints,
               SUM(c.crash) AS crashes, SUM(c.fire) AS fires,
               SUM(c.injuries) AS injuries, SUM(c.deaths) AS deaths
          FROM complaint_components cc
          JOIN complaints c ON c.odi_number = cc.odi_number
         WHERE cc.vehicle_id = ?
         GROUP BY cc.component
         ORDER BY complaints DESC""", (vehicle_id,))


def campaigns(vehicle_id):
    """The recall campaigns NHTSA opened on this vehicle, oldest first."""
    return sql("""SELECT campaign_number, component_head, component, consequence
                    FROM recalls
                   WHERE vehicle_id = ?
                   ORDER BY campaign_number""", (vehicle_id,))

time: 626 µs (started: 2026-09-26 21:06:47 +05:30)


In [13]:
# One complaint, whole. This is the unit of work.
r = narratives("chevrolet-bolt-ev-2020", limit=1, chars=700)[0]
print(f"ODI {r['odi_number']}  filed {r['date_filed']}  ({r['components']})")
print(f"crash={r['crash']} fire={r['fire']} injuries={r['injuries']} deaths={r['deaths']}\n")
print(r["narrative"])


ODI 11683606  filed 08/28/2025  (ELECTRICAL SYSTEM,TIRES,ENGINE)
crash=0 fire=1 injuries=0 deaths=0

The contact owns a 2020 Chevrolet Bolt EV. The contact stated that while the vehicle was parked unattended, the vehicle exploded. The contact believed the failure was due to the battery. The front left tire message was displayed. The battery was previously replaced. The local dealer was contacted regarding the battery purchase. The vehicle was not diagnosed or repaired. The contact called another local dealer. La Quinta Chevy Cady Service, 79225 CA-111, La Quinta, CA 92253, to obtain the service records, but the vehicle was not diagnosed or repaired. The fire department was able to extinguish the fire. There were no reported injuries, police report filed, or airbag deployments. The manufactu
time: 1.37 ms (started: 2026-09-26 21:07:02 +05:30)


In [15]:

# And the tally for the same vehicle, counted by SQL. This is what the analysts are handed.
print()
for c in components("chevrolet-bolt-ev-2020"):
    print(f"  {c['component']:24s} {c['complaints']:4d} complaints   {c['crashes']} crashes  "
          f"{c['fires']} fires  {c['injuries']} injuries  {c['deaths']} deaths")


  ELECTRICAL SYSTEM          93 complaints   1 crashes  4 fires  0 injuries  0 deaths
  FUEL/PROPULSION SYSTEM     30 complaints   0 crashes  0 fires  0 injuries  0 deaths
  STEERING                   22 complaints   0 crashes  0 fires  0 injuries  0 deaths
  UNKNOWN OR OTHER           21 complaints   1 crashes  0 fires  0 injuries  0 deaths
  POWER TRAIN                11 complaints   1 crashes  0 fires  0 injuries  0 deaths
  ENGINE                      8 complaints   0 crashes  1 fires  0 injuries  0 deaths
  VEHICLE SPEED CONTROL       8 complaints   3 crashes  0 fires  0 injuries  0 deaths
  AIR BAGS                    6 complaints   3 crashes  0 fires  0 injuries  0 deaths
  SEAT BELTS                  6 complaints   0 crashes  0 fires  0 injuries  0 deaths
  EXTERIOR LIGHTING           5 complaints   0 crashes  0 fires  0 injuries  0 deaths
  LANE DEPARTURE              5 complaints   0 crashes  0 fires  0 injuries  0 deaths
  STRUCTURE                   5 complaints   0 crashe

In [16]:
class Finding(BaseModel):
    """One line of the brief: one component on one vehicle. Everything that crosses an agent
    boundary is one of these.

    The point is the size: an analyst reads tens of thousands of tokens of narrative and
    is allowed to return this much.
    """

    vehicle_id: str
    component: str = Field(
        description="Exactly one component, spelled as the data spells it, never a combination")
    failure: str = Field(
        description="The main failure in that component, in one sentence a safety engineer "
                    "would recognise")
    complaints: int = Field(
        description="How many of this vehicle's complaints are filed under this component")
    worst_harm: Literal["none", "injury", "crash", "fire"]
    evidence: list[int] = Field(
        description="ODI numbers, at most five, of complaints filed under this component that "
                    "show the failure")
    call: Literal["escalate", "monitor", "close"]
    reasoning: str = Field(description="Two sentences at most")


class Brief(BaseModel):
    """What the desk publishes."""

    headline: str
    findings: list[Finding]

time: 228 ms (started: 2026-09-26 21:11:32 +05:30)


In [17]:
from langchain_core.callbacks import get_usage_metadata_callback


def tokens(usage):
    """(input tokens, output tokens) counted inside one callback block, every model together."""
    counts = usage.usage_metadata.values()
    return (sum(c["input_tokens"] for c in counts),
            sum(c["output_tokens"] for c in counts))


def show_usage(usage, wall):
    """One line per model that ran inside the block, then the wall clock."""
    for model, c in usage.usage_metadata.items():
        print(f"{model:24s} {c['input_tokens']:>8,} in  {c['output_tokens']:>7,} out")
    print(f"{'wall clock':24s} {wall:>8.1f}s")

time: 150 ms (started: 2026-09-26 21:11:57 +05:30)


# How to Evaluate

In [18]:
def what_is_wrong(f):
    """Why the database contradicts this Finding, or "" if it does not. SQL decides, not a model."""
    # The ODI numbers of every complaint filed under this component on this vehicle. All three
    # checks below compare the Finding against this one set.
    filed = {row["odi_number"] for row in sql(
        "SELECT odi_number FROM complaint_components WHERE vehicle_id = ? AND component = ?",
        (f.vehicle_id, f.component), limit=5000)}

    # Check 1: the component exists on this vehicle. A name the model invented, such as
    # "SERVICE BRAKES / AIR BAGS", matches no row, so the set is empty.
    if not filed:
        return f"component '{f.component}' has no complaints on this vehicle"

    # Check 2: the complaint count is the database's count. Both directions count as wrong:
    # calling a 407-complaint component 0 is not caution.
    if f.complaints != len(filed):
        return f"claims {f.complaints} complaints, database has {len(filed)}"

    # Check 3: every cited report really is filed under this component. A brake complaint
    # cited as evidence for an airbag finding supports nothing.
    not_filed = [odi for odi in f.evidence if odi not in filed]
    if not_filed:
        return f"cites reports that are not {f.component} on this vehicle: {not_filed}"

    # All three passed. What was never checked: the `failure` sentence.
    return ""


def coverage(vehicle_id, ranked, k=3):
    """Of the desk's top-k components for this vehicle, the share NHTSA went on to recall —
    next to the base rate, the share of all this vehicle's components that were recalled."""
    # The components that have complaints on this vehicle: the only ones a desk could rank.
    present = {c["component"] for c in components(vehicle_id)}
    # Of those, the ones NHTSA opened a recall on (a recall's `component_head` is the name the
    # complaints use).
    recalled = {c["component_head"] for c in campaigns(vehicle_id)} & present
    # The desk's top k, in the order it listed them.
    top = ranked[:k]
    return {
        # precision: the share of the top k that were recalled (0 if the desk named nothing)
        "precision": sum(c in recalled for c in top) / len(top) if top else 0.0,
        # base rate: what picking this vehicle's components at random would score
        "base_rate": len(recalled) / len(present),
    }


def run_scoreboard(name, design):
    """Run one design, print everything it is judged on, and return it as one row.

    `design` is any function that takes no arguments and returns a list of Findings.
    """
    # 1. Run the design and time it. Every model call made inside the `with` block is added
    #    to `usage`, including calls made by agents running in parallel.
    t0 = time.time()
    with get_usage_metadata_callback() as usage:
        findings = design()
    wall = time.time() - t0

    # 2. Check every Finding against the database and print one line for each: ✅ if SQL
    #    agrees with it, ❌ and the reason on the next line if not. n= is the count it claims.
    print(f"\n{name}: {len(findings)} findings on "
          f"{len({f.vehicle_id for f in findings})} of {len(WATCHLIST)} vehicles\n")
    wrong = 0
    for f in findings:
        why = what_is_wrong(f)  # "" when the database agrees
        print(f"  {'❌' if why else '✅'} {f.vehicle_id:24s} {f.component[:24]:24s} "
              f"n={f.complaints:<5} {f.worst_harm:6s} {f.call:8s} {f.failure[:40]}")
        if why:
            wrong += 1
            print(f"       {why[:100]}")

    # 3. The ranking: for each vehicle, its components in the order the design listed them.
    rankings = {v: [f.component for f in findings if f.vehicle_id == v] for v in WATCHLIST}

    # p@3: the share of each vehicle's top three that NHTSA later recalled, averaged over all
    # eight vehicles. `guessing` is the same average for picking components at random.
    scores = [coverage(v, ranked) for v, ranked in rankings.items()]
    p_at_3 = sum(s["precision"] for s in scores) / len(scores)
    guessing = sum(s["base_rate"] for s in scores) / len(scores)

    # Same order as COUNT(*): the number of vehicles whose ranking is exactly "most complaints
    # first", which a plain GROUP BY gives you with no model at all.
    same_order = sum(1 for v, ranked in rankings.items()
                     if ranked and ranked == [c["component"] for c in components(v)][:len(ranked)])

    # 4. Tokens, added up across every model that ran inside the block.
    in_tokens, out_tokens = tokens(usage)

    # 5. The summary, then everything returned as one row for the side-by-side table in P12.
    print(f"\n{'wrong findings':24s} {wrong:>8} of {len(findings)}")
    print(f"{'p@3 against the recalls':24s} {p_at_3:>8.2f}   (guessing scores {guessing:.2f})")
    print(f"{'same order as COUNT(*)':24s} {same_order:>8} of {len(WATCHLIST)} vehicles")
    show_usage(usage, wall)
    return {"name": name, "findings": findings, "wrong": wrong, "p_at_3": p_at_3,
            "same_order": same_order, "in_tokens": in_tokens, "out_tokens": out_tokens,
            "wall": wall}

time: 2.7 ms (started: 2026-09-26 21:12:39 +05:30)


# One Agent

In [19]:
from langchain.agents import create_agent
from langchain.tools import tool

READS = {"sql": 0, "narrative_calls": 0, "narratives_read": 0}


@tool
def run_sql(query: str) -> str:
    """Run a read-only SQL query over the field reports.

    Tables:
      watchlist(vehicle_id, make, model, model_year, complaint_count)
      complaints(odi_number, vehicle_id, date_filed, components, crash, fire,
                 injuries, deaths, narrative)
      complaint_components(odi_number, vehicle_id, component)
    """
    READS["sql"] += 1
    if "recall" in query.lower():
        # The recalls, and the watchlist's recall_count, are what the scoreboard grades
        # against. No agent reads them.
        return "SQL error: this desk has no access to recall data"
    try:
        # SELECT * never names recall_count, so drop it from whatever comes back.
        rows = [{k: v for k, v in row.items() if k != "recall_count"} for row in sql(query)]
        return json.dumps(rows)[:4000]
    except Exception as exc:
        return f"SQL error: {exc}"


@tool
def read_complaints(vehicle_id: str, component: str = "", limit: int = 20) -> str:
    """Read complaint narratives for one vehicle, worst reported harm first."""
    READS["narrative_calls"] += 1
    # max(..., 1) matters: sqlite's fetchmany(-1) returns every row, and while this notebook
    # was being built one run asked for limit=-1 and read all 685 narratives of one vehicle.
    rows = narratives(vehicle_id, component or None, limit=min(max(limit, 1), 40), chars=600)
    READS["narratives_read"] += len(rows)
    return json.dumps(rows)

time: 2.83 s (started: 2026-09-26 21:14:28 +05:30)


In [28]:
from langchain.agents.structured_output import ToolStrategy
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError

BRIEF_TASK = """You are the Canary field-safety desk at Halyard Analytics.

Produce this week's field-safety brief covering every vehicle on the watchlist. For each
vehicle, return its three most important component-level safety findings, most dangerous
first. A finding is one component, named exactly as the data names it, never a combination.
For each, give the main failure in that component in one sentence, how many of the vehicle's
complaints are filed under that component, the worst harm seen, up to five ODI numbers of
complaints filed under it as evidence, and a call of escalate / monitor / close.

Every number you write must come from the data. Cover all eight vehicles."""

solo = create_agent(
    ChatOpenAI(model=JUDGE, reasoning_effort=EFFORT),
    tools=[run_sql, read_complaints],
    system_prompt="You are a vehicle defect analyst. Be precise with numbers.",
    # If the final brief breaks the schema (a worst_harm of "death", say), the agent is shown
    # the validation error and tries again, instead of the run failing on its very last step.
    response_format=ToolStrategy(Brief),
)

time: 4.14 ms (started: 2026-09-26 21:18:41 +05:30)


In [29]:
from langchain_core.messages import AIMessage, ToolMessage


def show_trace(messages):
    """Print what the agent did, step by step: each tool it asked for (→), and the start of
    what came back (←)."""
    step = 0
    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            step += 1
            for call in message.tool_calls:
                arguments = ", ".join(repr(value) for value in call["args"].values())
                print(f"{step:3d} → {call['name']}({arguments[:220]})")
        elif isinstance(message, ToolMessage):
            print("    ← " + message.content.replace("\n", " ⏎ ")[:140])

time: 439 µs (started: 2026-09-26 21:18:55 +05:30)


In [30]:
SOLO_STEP_BUDGET = 40


def one_agent():
    """P1's design: one agent does the whole job. Prints its steps, returns its Findings."""
    try:
        result = solo.invoke({"messages": [{"role": "user", "content": BRIEF_TASK}]},
                             config={"recursion_limit": SOLO_STEP_BUDGET})
    except GraphRecursionError:
        print(f"the agent used all {SOLO_STEP_BUDGET} steps without producing a brief")
        return []
    show_trace(result["messages"])
    return result["structured_response"].findings


solo_run = run_scoreboard("one agent (P1)", one_agent)

  1 → run_sql('select vehicle_id, make, model, model_year, complaint_count from watchlist order by vehicle_id;')
    ← [{"vehicle_id": "chevrolet-bolt-ev-2020", "make": "chevrolet", "model": "bolt ev", "model_year": 2020, "complaint_count": 172}, {"vehicle_id
  2 → run_sql('select distinct component from complaint_components order by component;')
    ← [{"component": "AIR BAGS"}, {"component": "BACK OVER PREVENTION"}, {"component": "CARRY HANDLE, SHELL, BASE"}, {"component": "CHEST CLIP, BU
  3 → read_complaints('chevrolet-bolt-ev-2020', 5)
    ← [{"odi_number": 11683606, "date_filed": "08/28/2025", "components": "ELECTRICAL SYSTEM,TIRES,ENGINE", "crash": 0, "fire": 1, "injuries": 0, 
  4 → read_complaints('ford-f-150-2021', 5)
    ← [{"odi_number": 11683058, "date_filed": "08/26/2025", "components": "AIR BAGS", "crash": 1, "fire": 0, "injuries": 2, "deaths": 0, "narrativ
  5 → read_complaints('honda-accord-2019', 5)
  5 → read_complaints('hyundai-elantra-2021', 5)
  5 → read_complaint

# Part 2

```mermaid
flowchart LR
    L["for loop<br/>code decides<br/>what runs next"] -->|"one vehicle<br/>at a time"| A["analyst<br/>gpt-5-nano<br/>reads ONE vehicle"]
    D[("field_reports.db")] -.->|"narratives<br/>and SQL counts"| A
    A -->|"Finding objects"| E["editor<br/>gpt-5-mini<br/>reads no data"]
    E --> B(["the brief"])

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef python fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333

    class A,E model
    class L python
    class D stored
    class B endpoint
```

```python
class Finding(BaseModel):
    vehicle_id: str
    component:  str        # exactly one component, as the data spells it
    failure:    str        # the main failure in that component, one sentence
    complaints: int        # complaints filed under that component
    worst_harm: Literal["none", "injury", "crash", "fire"]
    evidence:   list[int]  # ODI numbers filed under that component
    call:       Literal["escalate", "monitor", "close"]
    reasoning:  str
```

In [32]:
components("chevrolet-bolt-ev-2020")[:3]

[{'component': 'ELECTRICAL SYSTEM',
  'complaints': 93,
  'crashes': 1,
  'fires': 4,
  'injuries': 0,
  'deaths': 0},
 {'component': 'FUEL/PROPULSION SYSTEM',
  'complaints': 30,
  'crashes': 0,
  'fires': 0,
  'injuries': 0,
  'deaths': 0},
 {'component': 'STEERING',
  'complaints': 22,
  'crashes': 0,
  'fires': 0,
  'injuries': 0,
  'deaths': 0}]

time: 2.22 ms (started: 2026-09-26 21:40:19 +05:30)


In [33]:
ANALYST_PROMPT = """You are one analyst on the Canary field-safety desk. This vehicle is your
whole assignment: {vehicle_id}.

Complaint counts per component, already tallied from the database. These are correct — use
them, do not recompute them:
{counts}

The {n} complaint narratives carrying the worst reported harm:
{narratives}

Return the three most important component-level safety findings for this vehicle.

Rules:
- `component` MUST be copied exactly from the counts table above. One component, never a list.
- `failure`: the main failure within that component, as the narratives describe it.
- `complaints` MUST be the count for that component from the table above.
- `evidence`: only ODI numbers that appear in the narratives above.
- `call`: escalate if that component has a recorded death or fire; otherwise monitor; close
  if the narratives show no real defect.
- Skip UNKNOWN OR OTHER unless the narratives show one specific failure behind it.
- Rank by danger, not by complaint count."""

def analyst_prompt(vehicle_id, limit=40, chars=600):
    counts = components(vehicle_id)
    rows = narratives(vehicle_id, limit=limit, chars=chars)
    return ANALYST_PROMPT.format(
        vehicle_id=vehicle_id,
        n=len(rows),
        counts="\n".join(
            f"  {c['component']}: {c['complaints']} complaints, {c['crashes']} crashes, "
            f"{c['fires']} fires, {c['injuries']} injuries, {c['deaths']} deaths"
            for c in counts),
        narratives="\n".join(
            f"  [{r['odi_number']}] ({r['components']}) crash={r['crash']} fire={r['fire']} "
            f"injuries={r['injuries']} deaths={r['deaths']}: {r['narrative']}"
            for r in rows),
    )


print(analyst_prompt("chevrolet-bolt-ev-2020"))

You are one analyst on the Canary field-safety desk. This vehicle is your
whole assignment: chevrolet-bolt-ev-2020.

Complaint counts per component, already tallied from the database. These are correct — use
them, do not recompute them:
  ELECTRICAL SYSTEM: 93 complaints, 1 crashes, 4 fires, 0 injuries, 0 deaths
  FUEL/PROPULSION SYSTEM: 30 complaints, 0 crashes, 0 fires, 0 injuries, 0 deaths
  STEERING: 22 complaints, 0 crashes, 0 fires, 0 injuries, 0 deaths
  UNKNOWN OR OTHER: 21 complaints, 1 crashes, 0 fires, 0 injuries, 0 deaths
  POWER TRAIN: 11 complaints, 1 crashes, 0 fires, 0 injuries, 0 deaths
  ENGINE: 8 complaints, 0 crashes, 1 fires, 0 injuries, 0 deaths
  VEHICLE SPEED CONTROL: 8 complaints, 3 crashes, 0 fires, 0 injuries, 0 deaths
  AIR BAGS: 6 complaints, 3 crashes, 0 fires, 0 injuries, 0 deaths
  SEAT BELTS: 6 complaints, 0 crashes, 0 fires, 0 injuries, 0 deaths
  EXTERIOR LIGHTING: 5 complaints, 0 crashes, 0 fires, 0 injuries, 0 deaths
  LANE DEPARTURE: 5 complaints, 

In [48]:
class AnalystReport(BaseModel):
    findings: list[Finding]


analyst_llm = ChatOpenAI(model='gpt-5', reasoning_effort=EFFORT).with_structured_output(
    AnalystReport)


def analyst(vehicle_id):
    """Agent one. Reads one vehicle. Returns its Findings."""
    report = analyst_llm.invoke(analyst_prompt(vehicle_id))
    for f in report.findings:
        f.vehicle_id = vehicle_id
    return report.findings

time: 4.86 ms (started: 2026-09-26 22:13:05 +05:30)


In [35]:
analyst("chevrolet-bolt-ev-2020")

[Finding(vehicle_id='chevrolet-bolt-ev-2020', component='ELECTRICAL SYSTEM', failure='High-voltage battery/propulsion electrical failures leading to battery fires and smoke (multiple reports of battery-related ignition while parked or charging).', complaints=93, worst_harm='fire', evidence=[11433995, 11429891, 11683892, 11683606], call='escalate', reasoning='Documented fires from the electrical system indicate a serious thermal/battery hazard that has caused vehicle fires in multiple reports.'),
 Finding(vehicle_id='chevrolet-bolt-ev-2020', component='VEHICLE SPEED CONTROL', failure='Accelerator pedal stick/drive-by-wire issue causing unintended acceleration or loss of control.', complaints=8, worst_harm='crash', evidence=[11738205], call='monitor', reasoning='Repeated reports of sticking accelerator and uncommanded acceleration present a clear crash risk, though no fatalities or fires are reported in these entries.'),
 Finding(vehicle_id='chevrolet-bolt-ev-2020', component='ENGINE', f

time: 11.9 s (started: 2026-09-26 21:43:21 +05:30)


In [36]:
editor_llm = ChatOpenAI(model=JUDGE, reasoning_effort=EFFORT)


def editor(findings):
    """Agent two. Writes the brief. Note what it is given: Findings, and nothing else.

    It has no database connection and no tools, so every number it has came in on a Finding.
    """
    rows = "\n".join(
        f"- {f.vehicle_id} | {f.component} | {f.complaints} complaints | worst harm "
        f"{f.worst_harm} | {f.call} | {f.failure}" for f in findings)
    prompt = ("Write the opening paragraph of this week's Canary field-safety brief for the "
              "safety engineer who signs it. Lead with what is most dangerous. Use only these "
              f"findings, and quote no number that is not here.\n\n{rows}")
    return editor_llm.invoke(prompt).content

time: 795 µs (started: 2026-09-26 21:49:12 +05:30)


In [37]:
t0 = time.time()
with get_usage_metadata_callback() as usage:
    pair_findings = []
    for vehicle_id in WATCHLIST[:2]:
        pair_findings += analyst(vehicle_id)
    paragraph = editor(pair_findings)

show_usage(usage, time.time() - t0)
print()
print(paragraph)

gpt-5-nano-2025-08-07      12,948 in    3,349 out
gpt-5-mini-2025-08-07         366 in      446 out
wall clock                   30.1s

Most dangerous: Ford F-150 powertrain defect (312 complaints, escalate) — unstable shifting and unintended loss of drive, including downshifts, hard shifts, and torque delivery issues that can cause loss of control in traffic. Also of immediate concern: Ford F-150 forward collision avoidance failures (42 complaints, monitor) and inconsistent/failed air-bag deployment in the F-150 (5 complaints, monitor) — both categories report crashes with the potential for occupant harm. Honda Accord entries to monitor include fuel/propulsion system faults (187 complaints, monitor) with stalls, sudden loss of power and documented fire risk; forward collision avoidance/automatic braking anomalies (153 complaints, monitor) causing unexpected engagement or braking suppression; and air-bag non-deployment or failures (29 complaints, monitor) resulting in injured occupants

```mermaid
flowchart LR
    subgraph one["1 · Pipeline"]
        direction LR
        p1["extract"] --> p2["judge"] --> p3["write"]
    end
    subgraph two["2 · Supervisor"]
        direction TB
        s1{"supervisor"} --> s2["specialist A"]
        s1 --> s3["specialist B"]
        s2 -.->|"back"| s1
        s3 -.->|"back"| s1
    end
    subgraph three["3 · Map-reduce"]
        direction TB
        m1["split"] --> m2["worker"] & m3["worker"] & m4["worker"]
        m2 & m3 & m4 --> m5["reduce"]
    end
    one ~~~ two ~~~ three

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef python fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef review fill:#fff3e0,stroke:#ef6c00,color:#8a3800

    class p1,p3,s1,s2,s3,m2,m3,m4 model
    class m1,m5 python
    class p2 review

    style one fill:none,stroke:#9aa0a6,color:#80868b
    style two fill:none,stroke:#9aa0a6,color:#80868b
    style three fill:none,stroke:#9aa0a6,color:#80868b
```

```mermaid
flowchart LR
    subgraph four["4 · Hierarchy"]
        direction TB
        h1{"lead"} --> h2{"sub-lead"} & h3{"sub-lead"}
        h2 --> h4["worker"] & h5["worker"]
        h3 --> h6["worker"] & h7["worker"]
    end
    subgraph five["5 · Handoff network"]
        direction LR
        n1["agent A"] <--> n2["agent B"]
        n2 <--> n3["agent C"]
        n1 <--> n3
    end
    subgraph six["6 · Debate"]
        direction TB
        d1["proposer"] --> d3{"judge"}
        d2["opponent"] --> d3
    end
    four ~~~ five ~~~ six

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef review fill:#fff3e0,stroke:#ef6c00,color:#8a3800

    class h1,h2,h3,h4,h5,h6,h7,n1,n2,n3,d1,d2 model
    class d3 review

    style four fill:none,stroke:#9aa0a6,color:#80868b
    style five fill:none,stroke:#9aa0a6,color:#80868b
    style six fill:none,stroke:#9aa0a6,color:#80868b
```

| shape | who decides the next step | costs | fails by |
|---|---|---|---|
| **Pipeline** | nobody — it is fixed | 1x, predictable | being unable to adapt; an early mistake is final |
| **Supervisor** | a model, every hop | 1 extra call per hop | ping-pong between specialists; never terminating |
| **Map-reduce** | code, up front | N in parallel, ~1x wall-clock | needing to know the split in advance |
| **Hierarchy** | models, at several levels | grows fast | the lead losing track of what the sub-leads did |
| **Handoff network** | whichever agent holds the baton | unbounded | no one owning the outcome |
| **Debate** | a judge, at the end | 3x or worse | agreeing with itself confidently |

### The rule worth remembering

> **Use code to route what you can predict. Use a model to route only what you cannot.**

Splitting a watchlist of eight vehicles into eight jobs needs no intelligence — a `for` loop
knows how to do it, for free, identically every time. Deciding which specialist should handle
an unlabelled incoming report does need intelligence. Most systems that disappoint have spent
a model call on the first kind of decision.

### What this desk needs

Two shapes, stacked:

- **Map-reduce** for the weekly brief, because the eight vehicles are known in advance and
  independent of each other. That is P4.
- **Supervisor** for the part that is *not* known in advance — a report arriving off-schedule
  that has to reach the right specialist. That is P6.

And then P7, which is not a shape so much as a rule about who is allowed to check the work.

```mermaid
flowchart LR
    S(["START"]) -->|"Send"| A1["analyst<br/>chevrolet-bolt-ev-2020"]
    S -->|"Send"| A2["analyst<br/>tesla-model-3-2021"]
    S -->|"Send"| A3["analyst × 6 more<br/>one vehicle each"]
    A1 --> R[/"findings<br/>operator.add appends"/]
    A2 --> R
    A3 --> R
    R --> E(["END"])

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333

    class A1,A2,A3 model
    class R stored
    class S,E endpoint
```


In [49]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class DeskState(TypedDict):
    vehicles: list[str]
    # Without operator.add, eight parallel writes to `findings` overwrite one another and
    # seven analysts' work disappears silently.
    findings: Annotated[list[Finding], operator.add]


class AnalystTask(TypedDict):
    """The private input one analyst receives. One vehicle. Nothing else."""
    vehicle_id: str


def analyst_node(task: AnalystTask) -> dict:
    return {"findings": analyst(task["vehicle_id"])}


def fan_out(state: DeskState):
    """Code decides the split. No model call, no variance, free."""
    return [Send("analyst", {"vehicle_id": v}) for v in state["vehicles"]]


builder = StateGraph(DeskState)
builder.add_node("analyst", analyst_node)
builder.add_conditional_edges(START, fan_out, ["analyst"])
builder.add_edge("analyst", END)
desk = builder.compile()

time: 1.59 ms (started: 2026-09-26 22:13:10 +05:30)


```mermaid
flowchart LR
    S(["START"]) -.->|"fan_out<br/>one Send per vehicle"| A["analyst<br/>gpt-5-nano<br/>one vehicle per copy"]
    A --> E(["END"])

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333

    class A model
    class S,E endpoint
```

Create production grade docker image, Arize Phoenix Tracking traces, prometheus, and track it on grafana. For our use-case. 

In [50]:
def fan_out_team():
    """P4's design: eight analysts at once, their Findings merged by the reducer."""
    return desk.invoke({"vehicles": WATCHLIST, "findings": []},
                       config={"max_concurrency": 8})["findings"]


team_run = run_scoreboard("fan-out team (P4)", fan_out_team)
team_findings = team_run["findings"]


fan-out team (P4): 24 findings on 8 of 8 vehicles

  ✅ ford-f-150-2021          POWER TRAIN              n=312   fire   escalate 10R80 transmission suffers uncommanded d
  ✅ ford-f-150-2021          ELECTRICAL SYSTEM        n=128   fire   escalate Water intrusion at the under-cowl DCN/po
  ✅ ford-f-150-2021          STEERING                 n=41    crash  monitor  Electric power steering rack/PSCM failur
  ✅ honda-accord-2019        ENGINE                   n=189   fire   escalate Head-gasket failure on the 1.5T engine c
  ✅ honda-accord-2019        FUEL/PROPULSION SYSTEM   n=187   fire   escalate Low-pressure fuel pump failure causes en
  ✅ honda-accord-2019        FORWARD COLLISION AVOIDA n=153   injury monitor  Collision Mitigation Braking System (AEB
  ✅ tesla-model-3-2021       SEAT BELTS               n=15    injury escalate Front seat belt buckle can unlatch while
  ✅ tesla-model-3-2021       LANE DEPARTURE           n=31    injury escalate Emergency Lane Keeping/Autosteer can 